# Lecture 17
## Building clear figures with Matplotlib
## Miles Chen, PhD

**How do I turn an analysis table into a clear, accurate figure?**

Adapted from *Python Data Science Handbook* by Jake VanderPlas and the
Matplotlib documentation.

# From tables to figures

In lectures 13–16, we checked passenger records, summarized groups, joined
labels, and reshaped tables. Now we will decide how to display those results.

Our main tools are `plt.subplots()`, `ax.plot()`, `ax.scatter()`, `ax.hist()`,
and `ax.bar()`. We will label, compare, and save the figures we create.

For every plot, ask: **What does a mark represent? Which records contribute?
What can a reader conclude?** Extra examples follow the main lesson.

In [1]:
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

titanic = pd.read_csv("../data/titanic.csv")
print("Matplotlib version:", matplotlib.__version__)
print("Passenger records:", len(titanic))
titanic.head()

Matplotlib version: 3.10.9
Passenger records: 891


,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# Figure, Axes, and axis

A **Figure** is the whole canvas. An **Axes** is a plotting area within it,
with its own title, scales, and labels. An x-axis or y-axis is one component
of that Axes. Despite its name, `ax` can refer to one Axes object.

`fig, ax = plt.subplots()` creates a figure with one plotting area.
We draw and customize using methods on `ax`, then display with `plt.show()`.

In [ ]:
x = np.linspace(0, 2 * np.pi, 500)
y = np.sin(x)

fig, ax = plt.subplots(figsize=(6, 3.5), layout="constrained")
ax.plot(x, y)
ax.set_title("A familiar function")
ax.set_xlabel("x (radians)")
ax.set_ylabel("sin(x)")
ax.set_xlim(0, 2 * np.pi)
plt.show()

# A line connects observations in the supplied order

Connecting nearby x-values is useful for a function or an ordered sequence.
It can mislead when each row is a different person in arbitrary order.

Below, each pair represents a different fictional student. Compare what the
connecting line suggests with what the unconnected points show.

In [ ]:
hours = [1, 3, 2, 4]
scores = [60, 82, 70, 85]

fig, axes = plt.subplots(1, 2, figsize=(8, 3.2), sharey=True, layout="constrained")
axes[0].plot(hours, scores, marker="o")
axes[0].set_title("Connected in row order")
axes[1].scatter(hours, scores)
axes[1].set_title("One point per student")
for ax in axes:
    ax.set_xlabel("Study time (hours)")
axes[0].set_ylabel("Score (points)")
plt.show()

# Choose the plot from the question

| Question | Useful starting point | A mark represents |
| --- | --- | --- |
| How does a value change along an ordered sequence? | Line plot | A value connected to the next value |
| How are two measurements related? | Scatterplot | One observation |
| How is a numeric variable distributed? | Histogram | A count within a numeric interval |
| How do group summaries compare? | Bar chart | A supplied group value |

A histogram groups numeric values into intervals. A bar chart displays
separate categories. They answer different kinds of questions.

# An age histogram: decide which records contribute

These 891 records are a subset of Titanic passenger records, not a complete
list of everyone aboard. We have 714 recorded ages and 177 missing ages.

For an age distribution, use recorded ages and report the exclusion. Do not
remove those passengers from unrelated analyses of recorded survival.

In [ ]:
age_records = titanic.dropna(subset=["age"])
print("Recorded ages:", len(age_records))
print("Missing ages:", titanic["age"].isna().sum())

In [ ]:
age_fig, ax = plt.subplots(figsize=(6.5, 3.8), layout="constrained")
age_bin_counts, age_bin_edges, _ = ax.hist(
    age_records["age"], bins=np.arange(0, 85, 5),
    color="steelblue", edgecolor="white",
)
ax.set_title("Recorded passenger ages (n = 714; 177 ages missing)")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Passenger records")
print("Records represented by the bins:", int(age_bin_counts.sum()))
plt.show()

# Bins change the appearance, not the observed ages

We supplied bin edges at 0, 5, 10, ..., 80. The last bin includes its right
endpoint. All recorded ages in this file lie within that range.

Compare 5-year and 10-year bins below. A wider bin can contain more records
simply because it covers more ages. Keep the same axis ranges when comparing
these displays, and avoid treating every small bump as a substantive pattern.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), sharex=True, sharey=True, layout="constrained")
for ax, width in zip(axes, [5, 10]):
    ax.hist(age_records["age"], bins=np.arange(0, 80 + width, width),
            color="steelblue", edgecolor="white")
    ax.set_title(f"{width}-year bins; n = {len(age_records)}")
    ax.set_xlabel("Age (years)")
axes[0].set_ylabel("Passenger records")
plt.show()

# A scatterplot: keep paired measurements together

To compare age and recorded fare, select rows that have **both** measurements.
Do not drop missing values separately from each Series: doing so can break
the pairing between a passenger's age and fare.

`alpha` controls transparency and `s` controls marker area. Smaller,
partly transparent points can make overlapping observations easier to see.

In [ ]:
age_fare = titanic.dropna(subset=["age", "fare"])
print("Records with both measurements:", len(age_fare))
print("Records excluded:", len(titanic) - len(age_fare))

scatter_fig, ax = plt.subplots(figsize=(6.5, 4), layout="constrained")
ax.scatter(age_fare["age"], age_fare["fare"], s=18, alpha=0.4, color="steelblue")
ax.set_title("Age and recorded fare: one point per passenger record")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Recorded fare (pounds)")
plt.show()

# Interpret the display, including its limits

Many records have relatively low fares, and several much larger fares
stretch the vertical scale. Overlapping points can hide the number of
records in a region. A scatterplot does not automatically calculate a
correlation or fit a regression line.

Cropping an axis hides values outside the view. Transforming a scale changes
how distances are represented. Either choice should be visible to the reader.
A relationship in these records does not establish a causal effect of age.

# Rebuild the familiar class/sex summary

We now return to **all** passenger records. An age is not required to
contribute to the survival summary.

The output has one row per class/sex group. The number of recorded survival
outcomes is the denominator of that group's mean 0/1 survival value.

In [ ]:
group_summary = (
    titanic.groupby(["pclass", "sex"], as_index=False, dropna=False)
    .agg(
        n_passengers=("passengerid", "size"),
        n_survival=("survived", "count"),
        survival_rate=("survived", "mean"),
        n_age=("age", "count"),
        mean_age=("age", "mean"),
    )
    .sort_values(["pclass", "sex"])
)
class_labels = pd.DataFrame({
    "pclass": [1, 2, 3], "class_name": ["First", "Second", "Third"]
})
analysis_table = group_summary.merge(
    class_labels, on="pclass", how="left", validate="many_to_one"
)
plot_table = analysis_table[
    ["pclass", "class_name", "sex", "n_passengers", "n_survival", "survival_rate"]
].sort_values(["pclass", "sex"]).reset_index(drop=True)
print("Group rows:", len(plot_table))
print("Passenger records represented:", plot_table["n_passengers"].sum())
plot_table

# `bar` draws the heights you supply

A bar can show a count, a rate, or another summary. Matplotlib's `ax.bar`
does not decide which summary to compute: we already made that decision.

Start with the three groups whose recorded sex is female. Each bar is a
group survival rate, not an individual passenger.

In [ ]:
female_summary = plot_table.loc[plot_table["sex"] == "female"]
fig, ax = plt.subplots(figsize=(6, 3.8), layout="constrained")
ax.bar(female_summary["class_name"], female_summary["survival_rate"], color="steelblue")
ax.set_title("Recorded survival rates: female passengers")
ax.set_xlabel("Ticket class")
ax.set_ylabel("Survival proportion")
ax.set_ylim(0, 1)
plt.show()

# Several panels share one Figure

`plt.subplots(1, 2)` returns a one-dimensional array of two Axes objects.
Use `axes[0]` and `axes[1]`, or loop over them.

A two-dimensional grid such as `plt.subplots(2, 3)` uses row/column indexing:
`axes[1, 1]` is the bottom-middle panel.

Shared axes make comparisons easier. In the next plot, every bar uses the
same scale with ticks from 0 to 1 and a little space above for labels.
The labels show the denominator of each rate.

In [ ]:
survival_fig, survival_axes = plt.subplots(
    1, 2, figsize=(9, 4.2), sharey=True, layout="constrained"
)
for ax, sex in zip(survival_axes, ["female", "male"]):
    subset = plot_table.loc[plot_table["sex"] == sex]
    bars = ax.bar(subset["class_name"], subset["survival_rate"], color="steelblue")
    ax.bar_label(bars, labels=[f"n={n}" for n in subset["n_survival"]], padding=3)
    ax.set_title(f"Recorded sex: {sex}")
    ax.set_xlabel("Ticket class")
    ax.set_ylim(0, 1.08)  # extra room for labels above the bars
    ax.set_yticks(np.arange(0, 1.01, 0.2))
survival_axes[0].set_ylabel("Survival proportion")
survival_fig.suptitle("Survival in the 891-record Titanic dataset")
plt.show()

# The caption is part of the analysis

**Example:** Among these 891 passenger records, recorded survival proportions
differ by ticket class and recorded sex; the third-class female group has
72 survivors among 144 recorded outcomes (0.50). All recorded survival
outcomes contribute, including records with missing ages; these descriptive
comparisons do not establish causal effects of class or sex.

The labels `n=...` count recorded survival outcomes, not survivors. There are
no error bars here: we are displaying the observed proportions in this file.

# Save the Figure you intend to share

`fig.savefig(...)` saves a specific Figure, even if you have created other
figures since then. Use readable labels at the intended output size.

The next cell creates a `figures` folder beside `live_demos` and saves the
survival figure as PNG and PDF. Rerunning the cell updates those two files.
PDF retains vector text and bars; `dpi` controls the raster PNG resolution.

In [ ]:
figure_dir = Path("../figures")
figure_dir.mkdir(exist_ok=True)
survival_fig.savefig(figure_dir / "17_survival_by_class.png", dpi=200, bbox_inches="tight")
survival_fig.savefig(figure_dir / "17_survival_by_class.pdf", bbox_inches="tight")
print("Saved the survival figure in:", figure_dir.resolve())

# Your turn: show how much age information is available

From `analysis_table`, calculate the percentage of missing ages in each
class/sex group. Create one bar-chart panel per recorded sex, using the same
percentage scale in both panels.

Include a title, units, and class labels. Write a two-sentence caption that
states the population, the denominator, and one comparison. Explain why a
missing age does not mean the passenger was a child or an adult.

In [ ]:
# Calculate the percentages, create the figure, and write your caption.

# One solution

The denominator is all passenger records in the group. We calculate missing
ages as the difference between row counts and recorded-age counts.

In [ ]:
age_availability = analysis_table.assign(
    missing_age_pct=lambda d: 100 * (d["n_passengers"] - d["n_age"]) / d["n_passengers"]
)
missing_fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True, layout="constrained")
for ax, sex in zip(axes, ["female", "male"]):
    subset = age_availability.loc[age_availability["sex"] == sex]
    bars = ax.bar(subset["class_name"], subset["missing_age_pct"], color="darkorange")
    ax.bar_label(bars, fmt="%.1f%%", padding=3)
    ax.set_title(f"Recorded sex: {sex}")
    ax.set_xlabel("Ticket class")
    ax.set_ylim(0, 40)
axes[0].set_ylabel("Missing ages (% of group records)")
missing_fig.suptitle("Age availability differs across passenger groups")
plt.show()

# An interpretation

Among these 891 passenger records, 42 of the 144 third-class female records
have no recorded age, or about 29.2%. Each percentage uses all records in
its class/sex group as the denominator; missing age indicates unavailable
information rather than an age category.

Before sharing any figure, check its row population, statistic, units,
scales, labels, and interpretation. Next time, seaborn will help map table
columns to plots and can also calculate summaries for us.

# colors, line styles, and markers

Repeated plotting calls add marks to the same Axes. Give groups labels and
call `ax.legend()`; use a line style or marker as well as color when helpful.

| Option | Examples |
| --- | --- |
| Named color | `"steelblue"`, `"darkorange"`, `"maroon"` |
| Color shortcuts | `b`, `g`, `r`, `c`, `m`, `y`, `k` (black), `w` (white) |
| Line style | `-` solid, `--` dashed, `-.` dash-dot, `:` dotted |
| Marker | `o` circle, `s` square, `d` diamond, `*` star, `x` cross |

[Named colors](https://matplotlib.org/stable/gallery/color/named_colors.html),
[line styles](https://matplotlib.org/stable/gallery/lines_bars_and_markers/linestyles.html),
[markers](https://matplotlib.org/stable/gallery/lines_bars_and_markers/marker_reference.html).

In [ ]:
x_line = np.linspace(0, 10, 100)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), layout="constrained")
for offset, style in enumerate(["-", "--", "-.", ":"]):
    axes[0].plot(x_line, x_line + offset, linestyle=style, label=style)
axes[0].set_title("Line styles and the default color cycle")
axes[0].legend(title="Style")
axes[1].plot(x, np.sin(x), color="steelblue", label="sin(x)")
axes[1].plot(x, np.cos(x), color="darkorange", linestyle="--", label="cos(x)")
axes[1].set_title("Two functions on one Axes")
axes[1].legend()
for ax in axes:
    ax.set_xlabel("x")
    ax.set_ylabel("Function value")
plt.show()

# aspect ratio and scatterplot encodings

Equal aspect makes one x-unit and one y-unit the same physical length. This
makes sense for comparable spatial coordinates; it is not a general rule
for variables with different units, such as age and fare.

In `scatter`, `s` specifies marker **area** in points squared. Numeric color
values can use a continuous colormap and colorbar. Separate categories are
often clearer with a discrete legend.

In [ ]:
rng = np.random.default_rng(21)
x_rand = rng.random(100)
y_rand = rng.random(100)
size_values = rng.integers(1, 11, 100)
color_values = rng.random(100)

fig, ax = plt.subplots(figsize=(5.5, 4.5), layout="constrained")
points = ax.scatter(x_rand, y_rand, s=30 * size_values, c=color_values,
                    cmap="viridis", alpha=0.6, marker="o")
ax.set_aspect("equal")
ax.set(xlabel="Simulated x", ylabel="Simulated y", title="Area and continuous color")
fig.colorbar(points, ax=ax, label="Simulated color variable")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4), layout="constrained")
ax.scatter(x_rand[:10], y_rand[:10], s=100, color="steelblue", marker="*", label="A")
ax.scatter(x_rand[10:20], y_rand[10:20], s=70, color="darkorange", marker="o", label="B")
ax.set(xlabel="Simulated x", ylabel="Simulated y", title="Separate calls can use separate markers")
ax.legend(title="Group")
plt.show()

# a two-dimensional array of subplots

`axes[row, column]` selects one panel. `axes.flat` lets us loop over all
panels without writing a nested loop. The bottom-middle panel is `[1, 1]`.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(8, 4), layout="constrained")
for i, ax in enumerate(axes.flat):
    ax.text(0.5, 0.5, f"Panel {i}", ha="center", va="center")
    ax.set_xticks([])
    ax.set_yticks([])
axes[1, 1].set_title("Bottom-middle panel")
plt.show()

# discontinuities and different scales

Tangent has vertical asymptotes. Restricting the y-axis alone does not stop
a line from connecting points across a discontinuity. Here we place `NaN`
near the asymptotes to break the drawn line.

The tangent panel uses different y-limits, so do not compare the apparent
heights across these panels without reading the scales.

In [ ]:
tangent = np.tan(x)
tangent[np.abs(np.cos(x)) < 0.05] = np.nan
fig, axes = plt.subplots(2, 2, figsize=(8, 5.5), layout="constrained")
axes[0, 0].plot(x, np.sin(x))
axes[0, 0].set_title("sin(x)")
axes[0, 1].plot(x, np.cos(x))
axes[0, 1].set_title("cos(x)")
axes[1, 0].plot(x, tangent)
axes[1, 0].set(title="tan(x), with breaks", ylim=(-5, 5))
axes[1, 1].plot(x, np.sin(x), label="sin(x)")
axes[1, 1].plot(x, np.cos(x), linestyle="--", label="cos(x)")
axes[1, 1].legend()
for ax in axes.flat:
    ax.set_xlabel("x (radians)")
    ax.set_ylabel("Function value")
plt.show()

# unequal panel sizes with GridSpec

A grid slice can make a plotting area span several rows or columns. Use
this when a figure needs a layout that a regular subplot array cannot give.

In [ ]:
fig = plt.figure(figsize=(9, 5), layout="constrained")
grid = fig.add_gridspec(2, 3)
ax1 = fig.add_subplot(grid[0, 0])
ax2 = fig.add_subplot(grid[0, 1:])
ax3 = fig.add_subplot(grid[1, :2])
ax4 = fig.add_subplot(grid[1, 2])
ax1.scatter(x_rand, y_rand, s=12)
ax1.set(xlabel="x", ylabel="y", title="Paired values")
ax2.hist(x_rand, bins=10)
ax2.set(xlabel="x", ylabel="Count", title="Distribution of x")
ax3.hist(y_rand, bins=10)
ax3.set(xlabel="y", ylabel="Count", title="Distribution of y")
ax4.plot(x_line, x_line ** 2)
ax4.set(xlabel="x", ylabel="x squared", title="A function")
plt.show()

# simulated histograms and density

`density=True` scales a histogram so that the total **area** is one. A bar's
height is a density, not the probability of observing that exact value.

Use the same bin edges when comparing groups. Each density below is
normalized separately, so it compares shapes rather than sample sizes.

In [ ]:
normal_sample = rng.normal(size=1000)
fig, axes = plt.subplots(1, 2, figsize=(8, 3), layout="constrained")
axes[0].hist(normal_sample)
axes[0].set_title("Default bins")
axes[1].hist(normal_sample, bins=25, color="steelblue", edgecolor="white")
axes[1].set_title("25 bins")
for ax in axes:
    ax.set(xlabel="Simulated value", ylabel="Count")
plt.show()

In [ ]:
samples = [rng.normal(0, 0.8, 10000), rng.normal(-2, 1, 10000), rng.normal(3, 2, 10000)]
shared_bins = np.linspace(min(a.min() for a in samples), max(a.max() for a in samples), 40)
fig, ax = plt.subplots(figsize=(7, 3.5), layout="constrained")
kwargs = dict(bins=shared_bins, density=True, histtype="stepfilled", alpha=0.35)
for values, label in zip(samples, ["mean 0, SD 0.8", "mean -2, SD 1", "mean 3, SD 2"]):
    ax.hist(values, label=label, **kwargs)
ax.set(xlabel="Simulated value", ylabel="Density", title="Three separately normalized distributions")
ax.legend()
plt.show()

# two-dimensional bins

When points overlap heavily, `hist2d` or `hexbin` can encode the number of
observations in each region with color. Read the colorbar as a count.

In [ ]:
x_norm, y_norm = rng.multivariate_normal([0, 0], [[1, 1], [1, 2]], 10000).T
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), layout="constrained")
hist2d_result = axes[0].hist2d(x_norm, y_norm, bins=30, cmap="Blues")
fig.colorbar(hist2d_result[3], ax=axes[0], label="Count in bin")
hexes = axes[1].hexbin(x_norm, y_norm, gridsize=30, cmap="Blues", mincnt=1)
fig.colorbar(hexes, ax=axes[1], label="Count in bin")
axes[0].set_title("Rectangular bins")
axes[1].set_title("Hexagonal bins")
for ax in axes:
    ax.set(xlabel="Simulated x", ylabel="Simulated y")
plt.show()

# contour and image displays of a function

`meshgrid` produces a grid of coordinates on which we evaluate a function.
Contour lines connect equal function values; an image encodes values by color.

`imshow` would otherwise label axes with array positions. We supply the
image's coordinate extent and put the smallest y-values at the bottom.

In [ ]:
x_grid = np.linspace(-3, 3, 101)
y_grid = np.linspace(-3, 3, 101)
X, Y = np.meshgrid(x_grid, y_grid)
Z = np.exp(-(X ** 2 + Y ** 2))
half_step = (x_grid[1] - x_grid[0]) / 2
extent = (-3 - half_step, 3 + half_step, -3 - half_step, 3 + half_step)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), layout="constrained")
contours = axes[0].contour(X, Y, Z)
axes[0].clabel(contours, fontsize=8)
axes[0].set_title("Contours")
image = axes[1].imshow(Z, origin="lower", extent=extent, cmap="viridis")
axes[1].set_title("Image with coordinate labels")
fig.colorbar(image, ax=axes[1], label="Function value")
for ax in axes:
    ax.set(xlabel="x", ylabel="y", aspect="equal")
plt.show()

# pandas plotting uses Matplotlib

A Series or DataFrame's `.plot()` can draw on an existing Axes using `ax=`.
You can then customize that Axes with the same methods used above.

The next examples use simulated cumulative sums and a fictional table of
industry counts. They illustrate plotting interfaces, not real trends.

In [ ]:
ts = pd.Series(rng.normal(size=1000), index=pd.date_range("2000-01-01", periods=1000)).cumsum()
walks = pd.DataFrame(rng.normal(size=(1000, 4)), index=ts.index, columns=list("ABCD")).cumsum()
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), layout="constrained")
ts.plot(ax=axes[0], title="One simulated random walk")
walks.plot(ax=axes[1], title="Four simulated random walks")
for ax in axes:
    ax.set_ylabel("Cumulative sum")
plt.show()

In [ ]:
industry_counts = pd.DataFrame(
    rng.integers(0, 10, size=(5, 4)), columns=["CA", "NY", "TX", "OH"],
    index=["Tech", "Agriculture", "Manufacturing", "Service", "Other"],
)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), layout="constrained")
industry_counts.plot.bar(ax=axes[0], rot=35)
axes[0].set(title="Fictional counts by industry", ylabel="Count")
industry_counts.T.plot.bar(stacked=True, ax=axes[1], rot=0)
axes[1].set(title="Same counts arranged by state", ylabel="Count")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

# stacked counts and proportions

A stacked count chart shows group sizes as well as composition. A normalized
chart makes every group's total height one, so keep its original row counts
available. `crosstab` counts combinations of categorical values.

Preserve `survived` as a numeric 0/1 variable in the original table. Rename
the columns of the plotting table for display instead.

In [ ]:
survival_counts = pd.crosstab(titanic["sex"], titanic["survived"])
survival_counts = survival_counts.rename(columns={0: "Did not survive", 1: "Survived"})
survival_proportions = survival_counts.div(survival_counts.sum(axis=1), axis=0)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), layout="constrained")
survival_counts.plot.bar(stacked=True, ax=axes[0], rot=0)
survival_proportions.plot.bar(stacked=True, ax=axes[1], rot=0)
axes[0].set(title="Counts retain group size", ylabel="Passenger records", xlabel="Recorded sex")
axes[1].set(title="Each group totals one", ylabel="Proportion within group", xlabel="Recorded sex", ylim=(0, 1))
for ax in axes:
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8)
plt.show()

# an age threshold with an explicit unknown group

Here 8 years is an illustrative threshold, not a universal definition of
childhood. Missing age is not evidence of being on either side of it.

With the float ages read from this CSV, `age < 8` is False for missing values.
A two-way label based only on that comparison would mix 177 unknown ages
into the older group. Assign a third label before plotting.

In [ ]:
age_group = pd.Series("Not recorded", index=titanic.index)
age_group.loc[titanic["age"] < 8] = "Under 8"
age_group.loc[titanic["age"] >= 8] = "8 or older"
print(age_group.value_counts())

age_survival_counts = pd.crosstab(age_group, titanic["survived"]).reindex(
    ["Under 8", "8 or older", "Not recorded"]
)
age_survival_proportions = age_survival_counts.div(age_survival_counts.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
age_survival_proportions.rename(columns={0: "Did not survive", 1: "Survived"}).plot.bar(
    stacked=True, ax=ax, rot=0
)
ax.set(xlabel="Recorded age group", ylabel="Proportion within group", ylim=(0, 1),
       title="Unknown ages remain a separate group")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
plt.show()

In [ ]:
# Two categorical row labels produce one row per sex/age-group combination.
sex_age_counts = pd.crosstab([titanic["sex"], age_group.rename("age_group")], titanic["survived"])
sex_age_proportions = sex_age_counts.div(sex_age_counts.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
sex_age_proportions.rename(columns={0: "Did not survive", 1: "Survived"}).plot.bar(
    stacked=True, ax=ax, rot=35
)
ax.set(xlabel="Recorded sex and age group", ylabel="Proportion within group", ylim=(0, 1))
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
plt.show()

# sums of die rolls

Each row below represents an independent simulation of five fair dice.
The columns contain cumulative sums of one through five dice.

More dice increase both the mean and the variance of the sum. The shape
begins to look more bell-shaped, but this display does not standardize the
sums and does not establish a normal approximation for every sample size.

In [ ]:
rolls = rng.integers(1, 7, size=(60000, 5))
df_dice = pd.DataFrame(rolls.cumsum(axis=1), columns=["one", "two", "three", "four", "five"])
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
df_dice.plot.hist(bins=np.arange(0.5, 31.5, 1), density=True, histtype="step", ax=ax)
ax.set(xlabel="Sum of die rolls", ylabel="Density (unit-width bins)", title="Sums of one through five dice")
ax.legend(title="Number of dice")
plt.show()

# References

- [Matplotlib quick start](https://matplotlib.org/stable/users/explain/quick_start.html)
- [Plot types](https://matplotlib.org/stable/plot_types/index.html)
- [Arranging subplots](https://matplotlib.org/stable/users/explain/axes/arranging_axes.html)
- [Saving figures](https://matplotlib.org/stable/api/_as_gen/matplotlib.figure.Figure.savefig.html)